# Monday: labeling text with models

Researchers label texts with models so they don't have to read every text. A label is a measurement, so you check it before you count it. Here you practice on r/AmItheAsshole posts, where the community's verdict is known for every post. You train a classifier on the posts from HW4, check it against a simple baseline, read its errors, and compare it with Gemini's labels for some of the same posts. Then you change Gemini's prompt yourself, try it on 12 posts, and plan how this would work on your project texts.

Work with a partner. Write code in the cells that start with a comment, and use the check comments to confirm your result. Switch who types after Part 5. Make a copy named `week10_live_workspace_mine.ipynb`.

In [ ]:
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", message="Unknown solver options")  # a harmless warning from LogisticRegression
pd.set_option("display.max_colwidth", 300)  # show more of each text

## 1. Load the posts and set a baseline

The same posts and the same filter as in HW4: posts with one of the four verdicts and a text that was not removed.

In [ ]:
VERDICT_LABELS = {
    "Not the A-hole": "NTA",
    "not the a-hole": "NTA",
    "Asshole": "YTA",
    "Everyone Sucks": "ESH",
    "No A-holes here": "NAH",
}

raw_posts = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/aita_top_subs.csv")
keep = (
    raw_posts["flair_text"].isin(VERDICT_LABELS)
    & raw_posts["selftext"].notna()
    & ~raw_posts["selftext"].isin(["[removed]", "[deleted]"])
)
posts = raw_posts.loc[keep, ["idstr", "title", "selftext", "flair_text"]].copy()
posts["verdict"] = posts["flair_text"].map(VERDICT_LABELS)
posts["text"] = posts["title"] + " " + posts["selftext"]  # the model reads the title and the post
posts["verdict"].value_counts()  # Check: NTA 3239, YTA 386, ESH 108, NAH 99

Before you judge a model, check how well you could do without one. The simplest baseline always guesses the most common verdict. Its accuracy is the share of posts with that verdict.

In [ ]:
# The accuracy of always guessing NTA. Check: 0.85

A model that never reads a post is right 85% of the time. What does that mean for an accuracy of 85%?

Your answer:

## 2. Split the posts

A model has to be judged on posts it did not learn from. `train_test_split` puts 75% of the posts in `train` and keeps 25% in `test`. `stratify` keeps the share of each verdict the same in both.

In [ ]:
train, test = train_test_split(posts, test_size=0.25, stratify=posts["verdict"], random_state=211)
len(train), len(test)  # Check: (2874, 958)

## 3. Train a classifier

The classifier has two parts. TF-IDF turns each post into a row of word weights. Logistic regression learns which weights go with which verdict. Fit TF-IDF on the training posts only, then use the same columns for the test posts.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", min_df=5)
train_rows = tfidf.fit_transform(train["text"])
test_rows = tfidf.____(test["text"])  # the same columns as train_rows
train_rows.shape, test_rows.shape  # Check: the same number of columns

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(train_rows, train["verdict"])  # learn from the training posts and their verdicts

`model.predict()` takes rows and returns one verdict for each. Predict the test posts and save the result in `test["prediction"]`.

In [ ]:
# Make test["prediction"] and show the accuracy. Check: 0.85

The classifier has the same accuracy as always guessing NTA. Count its predictions with `value_counts()`. What is it doing?

Your answer:

## 4. Look past accuracy

A confusion table counts each pair of true verdict (rows) and predicted verdict (columns). Correct predictions are on the diagonal.

In [ ]:
# Make a confusion table with pd.crosstab. Check: every prediction is in the NTA column

`class_weight="balanced"` makes mistakes on rare verdicts count more while the model learns, so it cannot win by always guessing NTA. Train a second model with it and save its predictions in `test["prediction_balanced"]`.

In [ ]:
# Train model_balanced, make test["prediction_balanced"], and show the accuracy. Check: 0.74

In [ ]:
pd.crosstab(test["verdict"], test["prediction_balanced"])  # Check: 34 YTA posts predicted YTA

In [ ]:
# Accuracy for each verdict
test["correct_balanced"] = test["prediction_balanced"] == test["verdict"]
test.groupby("verdict")["correct_balanced"].mean().round(2)  # Check: NTA 0.83, YTA 0.35

The balanced model has a lower accuracy. Is it worse? Which model would you use to find YTA posts?

Your answer:

## 5. Read the errors and the words

The model learned one weight per word for each verdict. These are the words that push a post most toward YTA.

In [ ]:
weights = pd.DataFrame(model_balanced.coef_.T, index=tfidf.get_feature_names_out(), columns=model_balanced.classes_)
weights["YTA"].sort_values(ascending=False).head(10)  # Check: girlfriend first

In [ ]:
# Show the 10 words that push a post most toward ESH.

What kind of words push a post toward ESH? Would they help with posts the model has not seen?

Your answer:

In [ ]:
# YTA posts that the balanced model called NTA
missed = test[(test["verdict"] == "YTA") & (test["prediction_balanced"] == "NTA")]
missed[["idstr", "title"]].head(10)  # Check: t3_oree89 first

Read the titles, and open two of the posts in `missed`. Why might a word-based model call them NTA?

Your answer:

## 6. Compare with Gemini

We asked Gemini (`gemini-3.5-flash-lite`) to predict the verdict of 40 test posts, 10 of each verdict, with the same system instruction you will use in HW5. Its labels are saved in a file. In Part 7 you call Gemini yourself.

In [ ]:
gemini = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/week10_gemini_aita_labels.csv")
gemini = gemini.merge(test[["idstr", "title", "prediction_balanced"]], on="idstr")
gemini[["idstr", "verdict", "llm_label", "prediction_balanced", "llm_reason"]].head()  # Check: 40 rows

Compute the accuracy of three methods on these 40 posts: always guessing NTA, the balanced classifier, and Gemini.

In [ ]:
# Three accuracies on the 40 posts. Check: NTA 0.25, classifier 0.35, Gemini 0.475

In [ ]:
# Make a confusion table for Gemini.

Always guessing NTA scored 0.85 on all test posts but 0.25 here. Why? Which verdicts does Gemini get right, and which does it miss? Read two of its reasons for a wrong label.

Your answer:

### Free text or a fixed form

Gemini's labels above came with a schema, which allowed only one of the four labels and a reason. We also asked Gemini about 8 of the posts in two other ways: with no instruction at all, only the post, and with the system instruction but no schema. `str.findall` lists every verdict word in each answer.

In [ ]:
free = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/week10_gemini_free_text.csv")
free["labels_found"] = free["answer"].str.findall(r"\b(?:NTA|YTA|ESH|NAH)\b")
free[["idstr", "verdict", "asked", "labels_found"]]  # Check: 16 rows, two for each post

In [ ]:
# Add free["n_words"], the number of words in each answer, and show the shortest and longest for each way of asking.
# Check: 138 to 642 words with no instruction, 22 to 40 with the instruction

In [ ]:
print(free.loc[2, "answer"])  # one answer with no instruction, in full

Which label would you record for the answer above? For which posts does the label change with how you ask? What would it take to get labels out of 3,000 answers like these?

Your answer:

## 7. Change the prompt

Gemini got 3 of the 10 ESH posts and 1 of the 10 NAH posts right. The first prompt names the four labels but never says what separates them. Write one change to the prompt and try it on 12 of the 40 posts.

This part calls Gemini, so you need `GEMINI_API_KEY` in your `.env` file. It makes 13 calls, which cost less than one cent.

In [ ]:
# Supplied: connect to Gemini. The key is read from your .env file and never printed.
import json
import logging
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env", override=True)
        break
key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"
logging.getLogger("google_genai.types").setLevel(logging.ERROR)  # a warning google-genai prints on every call
print("Connected to Gemini.")

In [ ]:
# Supplied: the first prompt, the schema, and a function that sends one post
FIRST_SYSTEM = """You read posts from the Reddit forum r/AmItheAsshole, where people describe a conflict and ask whether they were in the wrong.

Predict the verdict the Reddit community gave. Use one of these labels:
NTA (not the asshole), YTA (you're the asshole), ESH (everyone sucks here), NAH (no assholes here).

Also give a reason of one sentence."""

SCHEMA = {
    "type": "object",
    "properties": {
        "llm_label": {"type": "string", "enum": ["ESH", "NAH", "NTA", "YTA"]},
        "llm_reason": {"type": "string"},
    },
    "required": ["llm_label", "llm_reason"],
}


def build_prompt(title, text):
    return f"Title: {title}\n\n{text}"


def ask(prompt, system, max_tries=4):
    """Send one prompt with this system instruction. Return the label and the reason as a dictionary."""
    config = types.GenerateContentConfig(system_instruction=system, response_mime_type="application/json", response_schema=SCHEMA)
    wait = 10
    for attempt in range(1, max_tries + 1):
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt, config=config)
            time.sleep(6)  # stay under the free limit of calls per minute
            return json.loads(response.text)
        except errors.APIError as error:
            if error.code not in {429, 503} or attempt == max_tries:
                raise
            print(f"Error {error.code}. Waiting {wait} seconds, then trying again.")  # 429: too many calls, 503: server busy
            time.sleep(wait)
            wait = wait * 2

In [ ]:
# 12 of the 40 posts, 3 of each verdict. llm_label is the label the first prompt gave.
few = gemini.groupby("verdict").sample(n=3, random_state=211)
few = few.merge(posts[["idstr", "selftext"]], on="idstr")
few[["idstr", "verdict", "llm_label", "title"]]  # Check: 12 rows

In [ ]:
# Ask Gemini about the first post in few with FIRST_SYSTEM, and print the answer.
# Use ask(build_prompt(title, text), system). Check: a dictionary with llm_label and llm_reason

In [ ]:
# Copy FIRST_SYSTEM here and change one thing that could help with ESH and NAH.
MY_SYSTEM = """..."""

In [ ]:
# Label the 12 posts with your prompt: 12 calls, about 2 minutes
assert "..." not in MY_SYSTEM, "Write your prompt in MY_SYSTEM first."
answers = []
for post in few.itertuples():
    answer = ask(build_prompt(post.title, post.selftext), MY_SYSTEM)
    answers.append({"idstr": post.idstr, "my_label": answer["llm_label"], "my_reason": answer["llm_reason"]})
    print(len(answers), "of", len(few), "done")
few = few.drop(columns=["my_label", "my_reason"], errors="ignore").merge(pd.DataFrame(answers), on="idstr")

In [ ]:
# The accuracy of the first prompt (llm_label) and of yours (my_label) on the 12 posts.
# Check: the first prompt gets 0.42 (5 of 12)

In [ ]:
pd.crosstab(few["verdict"], few["my_label"])  # rows: true verdict, columns: your prompt's label

Did your change help on these 12 posts? Compare with the pairs next to you. Here one post is 8 percentage points.

Your answer:

### Our change: define each label

We tried one change, a definition for each label:

- NTA (not the asshole): the poster was not in the wrong, and someone else was.
- YTA (you're the asshole): the poster was in the wrong, and the others were not.
- ESH (everyone sucks here): the poster was in the wrong, and someone else was too.
- NAH (no assholes here): nobody was in the wrong. They disagree, but each has a fair point.

We asked Gemini about all 40 posts with this second prompt, and about 40 other test posts with both prompts. The second prompt was written after reading Gemini's errors on the same 40 posts. Nobody looked at the other 40.

In [ ]:
versions = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/week10_gemini_prompt_versions.csv")
versions["correct"] = versions["llm_label"] == versions["verdict"]
versions.groupby(["posts", "prompt"]).size()  # Check: 40 in each group

In [ ]:
# The accuracy of each prompt on each set of posts. Check: same 40: first 0.475, second 0.525; other 40: first 0.525, second 0.55

In [ ]:
# Accuracy for each verdict: one row for each set of posts and prompt
versions.groupby(["posts", "prompt", "verdict"])["correct"].mean().unstack()

Did the second prompt help? Is the change the same on the same 40 posts and on the other 40? Which number would you report, and why? The first prompt, run again a week later on the same 40 posts, got 22 right instead of 19. Is the change bigger than that?

Your answer:

In [ ]:
# Posts in the other 40 whose label changed between the two prompts
other = versions[versions["posts"] == "other 40"]
labels = other.pivot(index="idstr", columns="prompt", values="llm_label")
changed = labels[labels["first"] != labels["second"]]
changed.join(posts.set_index("idstr")[["verdict", "title"]])  # Check: 4 posts

Read two of these posts and Gemini's reasons for both labels (they are in `other`). Did the definitions fix the label, or move the error to another verdict?

Your answer:

## 8. An easier label: false positives and false negatives

Replies are easier to label than posts. The file below has 20 AITA replies with the verdict word removed, 10 NTA and 10 YTA, and the labels of five methods:

- `word_search`: YTA if the reply contains words such as *rude*, *apologize* or *should have*, otherwise NTA
- `classifier`: TF-IDF and logistic regression, trained on other replies
- `gemini_short`: Gemini with the prompt `Label this reply NTA or YTA.`
- `gemini_careful`: Gemini with a system instruction that says the verdict was removed and defines NTA and YTA
- `gemini_tone`: Gemini with the prompt `Is this comment positive or negative?`, where negative counts as YTA

In [ ]:
reply_labels = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/week10_reply_labels.csv")
METHODS = ["word_search", "classifier", "gemini_short", "gemini_careful", "gemini_tone"]
reply_labels[["verdict", "text"] + METHODS].head()  # Check: 20 rows in reply_labels

In [ ]:
# Print the accuracy of each method in METHODS.
# Check: word_search 0.55, classifier 0.35, gemini_short 0.8, gemini_careful 0.8, gemini_tone 0.6

Say you are looking for YTA replies. A method can make two kinds of mistakes:

- A **false positive** is an NTA reply labeled YTA: the method finds something that is not there.
- A **false negative** is a YTA reply labeled NTA: the method misses something that is there.

Accuracy adds the two together. In a confusion table, they are the two cells off the diagonal.

*Hint: the false positives of method `m` are the rows where `(reply_labels[m] == "YTA") & (reply_labels["verdict"] == "NTA")`.*

In [ ]:
pd.crosstab(reply_labels["verdict"], reply_labels["gemini_tone"])  # rows: true verdict, columns: label

In [ ]:
# For each method, count the false positives and the false negatives.
# Check: gemini_short 0 and 4, gemini_careful 2 and 2, gemini_tone 7 and 1

In [ ]:
# The share of replies each method labels YTA. The true share is 0.5.
(reply_labels[METHODS] == "YTA").mean()

In [ ]:
# The NTA replies the tone prompt labeled YTA, with Gemini's reason
reply_labels.loc[(reply_labels["verdict"] == "NTA") & (reply_labels["gemini_tone"] == "YTA"), ["text", "gemini_tone_reason"]]

`gemini_short` and `gemini_careful` have the same accuracy. Which kind of mistake does each make? Which would you use to estimate the share of YTA replies, and which to collect YTA replies to read? The classifier gets the share almost right with an accuracy of 0.35. How?

Your answer:

## 9. What to report

Write two or three sentences on what this evaluation shows and what it does not. Name the baseline, give accuracy for each verdict, and say how many posts the comparison with Gemini rests on and which posts you used to write the prompt.

Your answer:

## 10. Your project texts

Answer these for your project, with the 50 texts you label in HW4 in mind.

1. One text is:
2. The concept you want to measure:
3. Your labels, one sentence each, plus `unclear`:
4. Could a keyword rule find the labels? Name one word you would try.
5. The baseline: about what share of your texts has the most common label?
6. Which of your 50 texts would you use to write a prompt, and which to score it?

Your answers:

## If you have time

### A. Another split

The accuracy depends on which posts end up in `test`. Repeat the split with five other values of `random_state` and see how much the balanced model's accuracy changes.

In [ ]:
for seed in [1, 2, 3, 4, 5]:
    train_s, test_s = train_test_split(posts, test_size=0.25, stratify=posts["verdict"], random_state=seed)
    tfidf_s = TfidfVectorizer(stop_words="english", min_df=5)
    model_s = LogisticRegression(max_iter=1000, class_weight="balanced")
    model_s.fit(tfidf_s.fit_transform(train_s["text"]), train_s["verdict"])
    accuracy = (model_s.predict(tfidf_s.transform(test_s["text"])) == test_s["verdict"]).mean()
    print(seed, round(accuracy, 3))  # Check: between 0.70 and 0.75

If the accuracy on 958 posts moves this much between splits, what does that mean for an accuracy on 40 posts?

Your answer:

### B. Precision and recall

Two common summaries of the mistakes, for the label you are looking for (here YTA):

- **Precision**: of the replies labeled YTA, the share that are YTA.
- **Recall**: of the YTA replies, the share labeled YTA.

Compute both for each method in Part 8. Which method has the highest precision, and which the highest recall?

In [ ]:
# Precision and recall for YTA. Check: gemini_short 1.0 and 0.6, gemini_tone 0.56 and 0.9